<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
《<a href="https://mng.bz/lZ5B">从零构建推理模型</a>》一书的配套代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库： <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第3章：习题解答

本 Notebook 中使用的依赖包：

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # 由 reasoning_from_scratch 使用
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

&nbsp;
## 练习 3.1：补充更多测试用例

- 可添加的测试案例理论上无穷无尽。
- 下方列出了一些较有意思的示例。

In [ ]:
from reasoning_from_scratch.ch03 import (
    run_demos_table
)

more_tests = [
    # 不同的括号类型
    ("check_17", "[1, 2]", "(1, 2)", True),

    # 科学计数法
    ("check_18", "1e-3", "0.001", True),

    # 使用脱字符指数的代数化简
    ("check_19", "(-3)^2", "9", True),

    # Unicode 减号（U+2212）与 ASCII 连字符减号
    ("check_20", "−1", "-1", True),

]

run_demos_table(more_tests)

- 可以看到，除 `check_20` 外，其他测试均通过；`check_20` 把普通减号替换成了肉眼几乎无法分辨的 Unicode 减号
- 可以在 `normalize_text` 函数中的任意位置加入下面任意一行来修复此测试用例

```python
text = text.replace("−", "-")
# 或者
text = text.replace("\u2212", "-")
```

- 乍看之下，下面这个测试用例也很有意思：


In [ ]:
extra_tests_1 = [
    ("check_21", "Text around answer 3.", "3", True)
]

run_demos_table(extra_tests_1)

- 乍看之下，会觉得我们的代码无法处理这种带文字的用例，其实这是个设计不佳的测试。
- 实际上，`run_demos_table` 只用于测试 `grade_answer` 函数，不多也不少。
- `grade_answer` 从不会直接看到整段答复，而是在调用前就已经从文本中提取出纯粹的答案片段。

换句话说，如果想测试带文本的答案，就需要像下面这样调用测试：

In [ ]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate
)


extra_tests_2 = [
    ("check_21",
     extract_final_candidate("Text around answer 3."),
     "3", True)
]
run_demos_table(extra_tests_2)

&nbsp;
## 练习 3.2：计算平均回复长度

- 方案 A：可以在 `evaluate_math500_stream` 函数中加入以下代码：

```python
# ...
# 放在 `num_correct = 0` 下方
total_len = 0

# ...
# 放在 for i, row in enumerate(math_data, start=1): 循环内
# 放在 `gen_text = ...` 下方任意位置
total_len += len(tokenizer.encode(gen_text))

# ...
# 放在底部 return 语句之前的任意位置
avg_len = total_len / num_examples
print(f"Average length: {avg_len:.2f} tokens")
```

- 另一种做法是直接利用主章节运行 `evaluate_math500_stream` 时生成的 `.jsonl` 文件来统计回复长度。
- 首先像下面这样读取 `.jsonl`：

In [ ]:
import json
from pathlib import Path

WHICH_MODEL = "base"

dev_name = "mps"  # 例如 "cuda"、"cpu"

# 可能需要调整此路径：
local_path = Path(f"math500-{dev_name}.jsonl")
if not local_path.exists():
    raise FileNotFoundError(
        f"{local_path} not found. Run ch03_main.ipynb to create it."
    )

results = []
with open(local_path, "r") as f:
    for line in f:
        if line.strip():
            results.append(json.loads(line))

print("Number of entries:", len(results))


- 每条记录里有多个字段，不过我们只关心 `"generated_text"` —— 它保存了模型的完整答案：

In [ ]:
print(results[0].keys())

- 每条记录虽然包含多个字段，但我们只需要 `"generated_text"`，也就是模型完整回答所在的键：

In [ ]:
from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer
)

if WHICH_MODEL == "base":

    download_qwen3_small(
        kind="base", tokenizer_only=True, out_dir="qwen3"
    )
    tokenizer_path = Path("qwen3") / "tokenizer-base.json"
    tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

elif WHICH_MODEL == "reasoning":

    download_qwen3_small(
        kind="reasoning", tokenizer_only=True, out_dir="qwen3"
    )
    tokenizer_path = Path("qwen3") / "tokenizer-reasoning.json"
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_path,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=True,
    )

- 接着按下列方式计算平均长度，和直接修改 `evaluate_math500_stream` 得到的效果类似：

In [ ]:
total_len = 0

for item in results:
    num_tokens = len(tokenizer.encode(item["generated_text"]))
    total_len += num_tokens

avg_len = total_len / len(results)
print(f"Average length: {avg_len:.2f} tokens")

| 模式      | 设备  | 平均长度 | MATH-500 规模  |
|-----------|---------|----------------|----------------|
| 基础      | CPU     | 97.3           | 10             |
| 基础      | MPS     | 98.0           | 10             |
| 推理 | CPU     | 891.80         | 10             |
| 推理 | MPS     | 1159.30        | 10             |
|           |         |                |                |
| 基础      | CUDA    | 96.74          | 500            |
| 推理 | CUDA    | 1361.21        | 500            |


- 可以看到，正如预期，推理版模型的回答明显更长。

&nbsp;
## 练习 3.3：扩展或替换评估数据集

- 如果想在更大的数据集上评估，只需把 `math_data[:10]` 换成其他切片或更大的数量（最高 500）。

```python
num_correct, num_examples, acc = evaluate_math500_stream(  # 运行 MATH-500 评测
    model, tokenizer, device, 
    math_data=math_data[:10],          # 仅使用前 10 条样本做快速测试
    max_new_tokens=2048,               # 单条生成的最大 token 数
    verbose=False                      # 不打印详细生成过程
)

```

- 下表给出了不同数据集规模下的准确率（MATH-500 测试集已经打乱，因此没有再次随机打乱）。

| 模式      | 设备  | 准确率 | MATH-500 规模  |
|-----------|---------|----------|----------------|
| 基础      | CUDA    | 30.0%    | 10             |
| 基础      | CUDA    | 34.0%    | 50             |
| 基础      | CUDA    | 27.0%    | 100            |
| 基础      | CUDA    | 31.0%    | 200            |
| 基础      | CUDA    | 15.3%    | 500            |
|           |         |          |                |
| 推理 | CUDA    | 90.0%    | 10             |
| 推理 | CUDA    | 58.0%    | 50             |
| 推理 | CUDA    | 58.0%    | 100            |
| 推理 | CUDA    | 56.0%    | 200            |
| 推理 | CUDA    | 48.2%    | 500            |

- 上述结果可以看出，仅用前 10 个样本并不能代表在完整 500 道 MATH-500 题目上的表现。

- 另外，我们还可以仿照 MATH-500 自行构建一套新数据集。
- 本仓库已经附带一个 MATH-500 风格的数据集，只需在本章主代码里把文件名 `math500_test.json` 改成  `math_new50_exercise.json` (位于https://github.com/rasbt/reasoning-from-scratch/tree/main/ch03/01_main-chapter-code)
- 该数据集上基础版和推理版的表现为：
    - base：36.0%（18/50）
    - reasoning：80.0%（40/50）
- 这说明尽管 MATH-500 测试集可能在 Qwen3 训练时出现过，但模型在新题上依然保持类似表现，说明并未对原始 MATH-500 数据过度拟合。

&nbsp;
## 练习 3.4：尝试不同提示模板

- 我们可以改用章节中提到的另一种提示语，把“Question”替换为“Problem”，大致如下：

```python
def render_prompt(prompt):
    template = (
        "You are a helpful math assistant.\n"        # 角色与任务说明
        "Solve the problem and write the final result on a new line as:\n"
        "\\boxed{ANSWER}\n\n"                         # 要求用 \boxed{} 输出最终答案
        f"Problem:\n{prompt}\n\nAnswer:"             # 插入题目并提示开始作答
    )
    return template                                  # 返回完整提示词模板

```

- 采用该提示后，基础模型在 500 个样本上的准确率从 15.3% 提升到 31.2%。
- 相反，推理模型的性能从 50.8% 略降到 50.0%。
- 说明基础模型对提示格式极度敏感（可能在训练中记住了某些特定格式的 MATH-500 样本），而推理模型几乎不受影响。